In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm
from collections import Counter
import time

In [2]:
headers = {
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Version/4.0 Chrome/130.0.6723.58 Safari/537.36 (AirWatch Browser v21.09.0.9)'
        }


header_2 = {
    'User-Agent': 'Mozilla/5.0 (Windows; U; Windows NT 5.1; en-US; rv:1.8.1) Gecko/20061121 BonEcho/2.0'
}
url = 'https://empresite.eleconomista.es/Actividad/TECNOCUT-SL/'

header_3 = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.101 Safari/537.36'
}


In [3]:
csv_entries = pd.read_csv('construction_entries_todo.csv', encoding ='latin1', names = ['company names'])

In [4]:
def import_csv(name: str):
    company_list = pd.read_csv(name+".csv")

In [11]:
business_name = 'Tecnocut Sl'

In [5]:
def url_search_term(business_name: str):
    business_name = business_name.replace(",", "")
    business_name = business_name.replace(".", "")
    url_st = business_name.upper().replace(' ', '-')
    return url_st

In [176]:
r = requests.get(url,headers=headers, timeout=10)
print(str(r.status_code))
soup = BeautifulSoup(r.content)

200


In [20]:
hrefs = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))
hrefs[0]['href']

'https://empresite.eleconomista.es/TECNOCUT.html'

In [8]:
test = soup.find('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))
test['href']

'https://empresite.eleconomista.es/TECNOCUT.html'

In [26]:
soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text

'Tecnocut sl'

In [180]:
baseurl = 'https://empresite.eleconomista.es/Actividad/'


test_entry = csv_entries['company names'][0]
to_search = url_search_term(test_entry)
search_url = baseurl + to_search + '/'

r = requests.get(search_url,headers=headers, timeout=10)
print(str(r.status_code))

soup = BeautifulSoup(r.content)

test = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
test

200


'https://empresite.eleconomista.es/BRICOINSA.html'

In [6]:
def business_search(csv_entries: pd.DataFrame):
    headers = {
            'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Version/4.0 Chrome/130.0.6723.58 Safari/537.36 (AirWatch Browser v21.09.0.9)'
            }
    link_list = []
    baseurl = 'https://empresite.eleconomista.es/Actividad/'

    for company in tqdm(list(csv_entries['company names'])):
        search_term = url_search_term(company)
        search_url = baseurl + search_term + '/'
        try:
            r = requests.get(search_url,headers=header_2, timeout=30)
            soup = BeautifulSoup(r.content)
            # print(company)
            # print(str(r.status_code))
            try:
                if str(r.status_code)== '200':
                    first_result = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0].text
                    # if first_result.split(' ')[0]==company.split(' ')[0]:
                    url_results = soup.find_all('a', href=True, title=lambda value: value and value.startswith("Ver perfil de"))[0]['href']
                    link_list.append(url_results)
                elif str(r.status_code)== '429':
                    link_list.append('429 issue')
                else:
                    link_list.append('404 not found')
            except IndexError:
                link_list.append('no results found')
            time.sleep(2)
        except:
            link_list.append('requests issue')

    # links_found = list(set(link_list)).remove('404 not found')
    return link_list

In [7]:
link_list = business_search(csv_entries)

100%|██████████| 2419/2419 [2:06:24<00:00,  3.14s/it]    


In [21]:
company_info = csv_entries.copy()
company_info['found_links'] = pd.DataFrame(link_list, columns=['found_links'])
excluded_rows = company_info[(company_info['found_links']=='404 not found') | (company_info['found_links']=='429 issue') | (company_info['found_links']=='requests issue')].index
company_info.drop(index=excluded_rows, inplace=True)

In [ ]:
def cleanup_df(df: pd.DataFrame, link_list: list) -> None:
    df = csv_entries.copy()
    df['found_links'] = pd.DataFrame(link_list, columns=['found_links'])
    excluded_rows = df[(df['found_links']=='404 not found') | (df['found_links']=='429 issue') | (df['found_links']=='requests issue')].index
    df.drop(index=excluded_rows, inplace=True)
    return None

In [166]:
phonenumbers_found = []
urls_found = []
emails_found = []

for link in tqdm(company_info['found_links']):
    r = requests.get(link,headers=header_3, timeout=10)
    if r.status_code != 200:
        print(str(r.status_code))
    soup = BeautifulSoup(r.content)
    try:
        found_url = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline'))[0]['href']
    except:
        found_url='not found'
    urls_found.append(found_url)
    try:
        found_email = soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]
    except:
        found_email = 'not found'
    emails_found.append(found_email)
    try:
        found_phone = soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text
    except:
        found_phone = 'not found'
    phonenumbers_found.append(found_phone)
    time.sleep(3)

 10%|█         | 17/164 [01:03<09:26,  3.85s/it]

In [154]:
test_link = links_found[1]

In [155]:
r = requests.get(test_link,headers=header_3, timeout=10)
print(str(r.status_code))

soup = BeautifulSoup(r.content)

200


In [143]:
test_link

'https://empresite.eleconomista.es/DECORACIONES-IMCASA.html'

In [160]:
soup.find_all('a', href=True, target = '_blank', class_ = lambda value: value and value.startswith('text-bodytext-m text-secondary-800 underline email'))[0]['href'].split('?')[0]

'mailto:cjfranco@iprodat.es'

In [164]:
soup.find_all('span', class_=lambda value: value and value.startswith('text-bodytext-m text-neutrals-700 md:block hidden'))[0].text

'987238400'